# Görev D — Yazım-seyrekliği taraması (kapasite hipotezi testi)

**Soru:** §17 v2'de bellek 1024 token ötesinde şansa çöktü. Sebep unutma yasası mı,
yoksa **girişim/kapasite** mi (her 64 tokenda bir yazım → 16k'da ~250 rakip yazım,
32-boyutlu state'i doyuruyor)?

**Test:** Gap sabit (16384), yazım aralığı `LT_DIST_EVERY` ∈ {64, 256, 1024, 4096}.
Yani aynı mesafede rakip yazım sayısı 256 → 64 → 16 → 4'e iner. Eğitim yeniden
yapılmaz (checkpoint'ler §17 v2'den gelir), yalnızca eval — hızlıdır.

**ÖN-KAYITLI OKUMA (koşudan önce):**
- Seyreldikçe doğruluk **yükseliyorsa** → kapasite/girişim teşhisi doğrulanır;
  "O(1) bellek uzun ömür sağlar" iddiası **yazım-seyrekliği koşuluyla** ifade edilir.
- Doğruluk **her seyreklikte şansta kalıyorsa** → sorun kapasite değil, eğitim
  ufkunun (256) ötesine genelleme; sonraki adım eğitim ufkunu uzatmak.
- exp/cubic farkı yine ≤1 isabet ise, §15h/§17'nin "yasa fark etmiyor" sonucu pekişir.

Çalıştırma: GPU **T4**, Internet **On** → Run All (~15-20 dk).

In [ ]:
# --- 1. Repo + checkpointler (§17 v2 egitimi tekrar edilmez) ---
import subprocess, sys, os, itertools, json, re
os.chdir('/kaggle/working')
if not os.path.isdir('HFP'):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git'], check=True)
else:
    subprocess.run(['git','-C','HFP','pull'], check=True)
os.chdir('/kaggle/working/HFP')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — T4 secin!')
assert torch.cuda.is_available(), 'GPU secili degil'
ENV = dict(os.environ, PYTHONPATH='/kaggle/working/HFP')
# §17 v2 checkpointleri yoksa once egitilir (kol basi ~2-3 dk); varsa dogrudan eval.
print('mevcut ckpt:', [f for f in os.listdir('checkpoints') if f.startswith('lifetimev2')] if os.path.isdir('checkpoints') else 'yok')

In [ ]:
# --- 2. TARAMA: gap=16384 sabit, yazim araligi degisken ---
# Not: her kol once (gerekiyorsa) egitilir, sonra tek gap'te eval eder.
DIST = ['64', '256', '1024', '4096']
MODES = ['exp', 'cubic_flux_chunked']
SEEDS = ['0', '1', '2']
raw = {}
for de in DIST:
    for m in MODES:
        for s in SEEDS:
            env = dict(ENV, LT_GAPS='16384', LT_TRIALS='30', LT_DIST_EVERY=de)
            print(f'\n===== yazim/{de} token | {m} | seed {s} =====', flush=True)
            r = subprocess.run([sys.executable,'review_scripts/lifetime_retention.py', m, s, '900'],
                               check=True, env=env, capture_output=True, text=True)
            tail = r.stdout.strip().splitlines()[-3:]
            print('\n'.join(tail), flush=True)
            mm = re.search(r"16384: \(([\d.]+),", r.stdout)
            if mm: raw[(de, m, s)] = float(mm.group(1))
print('\nham sonuclar:', raw)

In [ ]:
# --- 3. OZET TABLO (seed-ortalama) + on-kayitli okuma ---
import statistics as st
print(f'{"yazim araligi":>14} | {"~rakip yazim":>12} | {"exp":>16} | {"cubic":>16}')
print('-'*70)
rows = []
for de in DIST:
    n_w = 16384 // int(de)
    cell = {}
    for m in MODES:
        vals = [raw[(de, m, s)] for s in SEEDS if (de, m, s) in raw]
        cell[m] = (st.mean(vals), min(vals), max(vals)) if vals else (float('nan'),)*3
    e, c = cell['exp'], cell['cubic_flux_chunked']
    rows.append((de, n_w, e, c))
    print(f'{de:>14} | {n_w:>12} | {e[0]:>6.1f}% ({e[1]:.0f}-{e[2]:.0f}) | {c[0]:>6.1f}% ({c[1]:.0f}-{c[2]:.0f})')
print('\nSans seviyesi: %3.3   (gap sabit 16384; sadece yazim yogunlugu degisiyor)')

best = max(rows, key=lambda r: max(r[2][0], r[3][0]))
worst = rows[0]
gain = max(best[2][0], best[3][0]) - max(worst[2][0], worst[3][0])
print(f'\nEn seyrek vs en yogun fark: {gain:+.1f} puan')
if gain >= 10:
    print('OKUMA: KAPASITE/GIRISIM teshisi DOGRULANDI -> "O(1) bellek uzun omur saglar"')
    print('       iddiasi YAZIM-SEYREKLIGI kosuluyla ifade edilmeli. Sonraki eksen:')
    print('       kapasite (dpfp nu, bulk_dim) ve yazim kapisi (gating).')
elif max(r[2][0] for r in rows) < 8 and max(r[3][0] for r in rows) < 8:
    print('OKUMA: Her seyreklikte sansta -> sorun kapasite DEGIL, egitim ufkunun (256)')
    print('       otesine genelleme. Sonraki adim: egitim ufkunu uzatmak (LT/CTX buyut).')
else:
    print('OKUMA: Kismi toparlanma — egilim var ama esik altinda; TRIALS artirilip tekrarlanmali.')
print('\nBu ciktiyi Claude\'a yapistirin; RESULTS §18 olarak islenecek.')